[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cstecker/politicsRLab/blob/main/In%20welcher%20Blase%20bin%20ich%20denn%20hier%20gelandet%20-%202026.ipynb)


# **In welcher Blase bin ich denn hier gelandet?**

Zum Einstieg in empirisches Arbeiten schauen wir uns eine Umfrage aus unserer eigenen Vorlesung an. Die Frage ist bewusst nah am Kurs: Wer sitzt hier eigentlich mit uns im Raum, und wie politisch ähnlich oder unterschiedlich sind wir?

Dabei geht es nicht darum, sofort "perfekt R zu können". Code ist hier eher ein Werkzeugkasten: Wir stellen eine Frage, geben R eine präzise Anweisung und schauen, ob die Antwort plausibel ist. Wenn etwas nicht funktioniert, ist das kein Drama, sondern Teil des Arbeitens mit Daten.


## Bevor wir loslegen

Führen Sie Code-Zellen mit **Strg + Enter** oder über den kleinen Play-Button aus. Sie können Code verändern und direkt sehen, was passiert.

**Datenschutz:** Der Datensatz für dieses Notebook ist anonymisiert. Er enthält keine Zeitstempel, kein Geburtsbundesland und keine offenen Freitextantworten. Sehr seltene Parteiangaben wurden zusammengefasst. Dadurch ist die Datei gut für Übungen geeignet, kann aber leicht von der internen Gesamtauswertung abweichen.


## 1. R vorbereiten

Zuerst laden wir das Paket **tidyverse**. Das ist eine Sammlung von R-Paketen, mit denen man Daten einlesen, sortieren, zusammenfassen und visualisieren kann.


In [ ]:
library(tidyverse)


## 2. Daten laden

Die anonymisierte Kursumfrage liegt auf GitHub. Die nächste Zelle lädt die Datei in Colab herunter und speichert sie unter dem Namen `survey2026-04_colab.rds`.


In [ ]:
download.file(
  "https://raw.githubusercontent.com/cstecker/politicsRLab/main/data/survey2026-04_colab.rds",
  "survey2026-04_colab.rds",
  mode = "wb"
)


Jetzt laden wir die Datei in R. Das Objekt nennen wir `survey`. Ab jetzt meint `survey` also unseren Datensatz.


In [ ]:
survey <- readRDS("survey2026-04_colab.rds")


Ein erster Blick: `glimpse()` zeigt uns, wie viele Zeilen und Spalten der Datensatz hat und welche Variablen darin vorkommen.


In [ ]:
glimpse(survey)


Aus der Excel-Welt kennen Sie rechteckige Tabellen: In den Zeilen stehen Fälle, in den Spalten Variablen. Hier ist ein kleiner Ausschnitt:


In [ ]:
head(survey)


## 3. Erste Frage: Wer sitzt im Raum?

Mit `count()` zählen wir, wie häufig einzelne Ausprägungen einer Variable vorkommen. Das ist oft der erste Schritt einer empirischen Analyse: nicht sofort kompliziert werden, sondern erst einmal verstehen, was in den Daten steht.


In [ ]:
survey |>
  count(geschlecht)


Die Zeichen `|>` heißen **Pipe**. Die Pipe nimmt das Ergebnis links davon und gibt es an den nächsten Befehl weiter. Gelesen wird das ungefähr so:

> Nimm `survey`, dann zähle die Werte der Variable `geschlecht`.

Probieren wir dasselbe mit einer politischen Einstellung. Die Variable `immi_self` misst Einstellungen zu Einwanderung auf einer Skala von 1 bis 11.


In [ ]:
survey |>
  count(immi_self)


## 4. Die Sonntagsfrage im Kurs

Nun wird es politikwissenschaftlich vertrauter: Welche Partei würden die Teilnehmenden wählen, wenn am nächsten Sonntag Bundestagswahl wäre?


In [ ]:
survey |>
  count(vote)


Sortiert ist die Tabelle leichter zu lesen. `sort = TRUE` stellt die häufigsten Antworten nach oben.


In [ ]:
survey |>
  count(vote, sort = TRUE)


Absolute Zahlen sind nützlich, aber meistens wollen wir zusätzlich Anteile sehen. Mit `mutate()` erzeugen wir neue Spalten: erst die Gesamtzahl der gültigen Antworten, dann den Prozentanteil.


In [ ]:
vote_share <- survey |>
  filter(!is.na(vote)) |>
  count(vote, sort = TRUE) |>
  mutate(
    total = sum(n),
    share = n / total,
    percent = round(share * 100, 1)
  )

vote_share


Und jetzt machen wir daraus ein Balkendiagramm. Das Paket `ggplot2` ist Teil des tidyverse. Die Grundidee ist:

- `ggplot()` startet die Grafik.
- `aes()` sagt, welche Variablen auf welche Achsen sollen.
- `geom_col()` zeichnet Balken.
- Mit `+` hängen wir weitere Grafikbefehle an.


In [ ]:
party_colors <- c(
  "CDU" = "#111111",
  "SPD" = "#E3000F",
  "Bündnis90 / Die Grünen" = "#46962B",
  "Die Linke" = "#BE3075",
  "FDP" = "#FFED00",
  "Ich würde nicht wählen" = "#9A9A9A",
  "Andere / selten genannt" = "#BDBDBD"
)

vote_share |>
  mutate(vote = forcats::fct_reorder(vote, share)) |>
  ggplot(aes(x = vote, y = share, fill = vote)) +
  geom_col() +
  coord_flip() +
  scale_y_continuous(labels = scales::label_percent(accuracy = 1)) +
  scale_fill_manual(values = party_colors, drop = FALSE) +
  labs(
    title = "Wahlabsicht in der Kursumfrage",
    x = NULL,
    y = "Anteil"
  ) +
  theme_minimal(base_size = 13) +
  theme(legend.position = "none")


Eine wichtige sozialwissenschaftliche Mini-Lektion steckt schon hier: Diese Grafik zeigt **nicht die Studierenden in Deutschland** und auch nicht **die Jugend**. Sie zeigt die Personen, die in dieser Vorlesung an dieser Umfrage teilgenommen haben. Daten brauchen immer Kontext.


## 5. Von Beschreibung zu Hypothese

Beschreiben ist der Anfang. Empirisches Arbeiten wird spannend, wenn wir Erwartungen formulieren und an Daten prüfen.

Eine klassische Frage der Wahlforschung lautet: Gibt es einen **Gender Gap** in politischen Einstellungen? Als einfache Hypothese könnten wir prüfen:

> Männliche und weibliche Studierende unterscheiden sich auf der Links-Rechts-Skala.

Die Variable `lire_self` reicht von 1 = links bis 11 = rechts. Mit einem Dichteplot vergleichen wir die Verteilungen.


In [ ]:
survey |>
  filter(
    geschlecht %in% c("Weiblich", "Männlich"),
    !is.na(lire_self)
  ) |>
  ggplot(aes(x = lire_self, color = geschlecht, fill = geschlecht)) +
  geom_density(alpha = 0.2, adjust = 1.4) +
  scale_x_continuous(breaks = 1:11, limits = c(1, 11)) +
  labs(
    title = "Links-Rechts-Selbsteinstufung nach Geschlecht",
    x = "1 = links, 11 = rechts",
    y = "Dichte",
    color = NULL,
    fill = NULL
  ) +
  theme_minimal(base_size = 13)


Der Plot beweist noch keine Theorie. Aber er hilft uns, genauer zu fragen: Sind die Unterschiede groß? Könnten sie zufällig sein? Welche Fälle fehlen wegen `NA`? Welche andere Erklärung wäre plausibel?


## 6. Zwei politische Dimensionen gleichzeitig

Politische Konflikte sind selten nur eindimensional. Unten steht jeder Punkt für eine Person. Die x-Achse zeigt Einstellungen zu Wirtschaft und Sozialstaat, die y-Achse Einstellungen zu Einwanderung. Die Punkte sind leicht verschoben (`geom_jitter()`), damit gleiche Antworten nicht exakt übereinanderliegen.


In [ ]:
survey |>
  filter(!is.na(econ_self), !is.na(immi_self), !is.na(vote)) |>
  ggplot(aes(x = econ_self, y = immi_self, color = vote)) +
  geom_jitter(width = 0.18, height = 0.18, alpha = 0.75, size = 2.4) +
  scale_x_continuous(breaks = 1:11, limits = c(1, 11)) +
  scale_y_continuous(breaks = 1:11, limits = c(1, 11)) +
  scale_color_manual(values = party_colors, drop = FALSE) +
  labs(
    title = "Wahlabsicht und politische Einstellungen",
    x = "1 = mehr Sozialstaat, 11 = weniger Sozialstaat",
    y = "1 = Zuzug erleichtern, 11 = Zuzug einschränken",
    color = "Wahlabsicht"
  ) +
  theme_minimal(base_size = 13)


Solche Grafiken sind kein Ersatz für Theorie. Sie sind ein Gesprächsangebot an die Theorie: Wo sehen wir Muster, wo Ausnahmen, wo neue Fragen?


## 7. Jetzt Sie

Suchen Sie sich eine Variable und stellen Sie eine kleine Frage. Gute Einstiegsfragen sind zum Beispiel:

- Welche Antworten sind besonders häufig?
- Gibt es Gruppenunterschiede?
- Passt das Muster zu einer politikwissenschaftlichen Erwartung?
- Welche Information fehlt, um die Frage wirklich gut zu beantworten?


In [ ]:
names(survey)


In [ ]:
# Hier könnte Ihr eigener Code stehen.
# Beispiele zum Umbauen:

survey |>
  count(klim_self)

# survey |>
#   group_by(vote) |>
#   summarise(mean_lire = mean(lire_self, na.rm = TRUE))
